# Cross-corpus evaluation — TCN+DBN on Vienna4x22 & Batik-plays-Mozart

Mazurka-trained TCN checkpoints tested **without re-training** on the two out-of-domain corpora packed by [CrossEval.ipynb](CrossEval.ipynb). For my PhD thesis.

- 5 mazurka folds × 2 corpora × 2 ckpt variants (`model_best.h5` and `model_final.h5`)
- Reuses the original TCN model + madmom DBN pipeline from `Eval_BeatTCN_MazurkaBL.ipynb` — only the dataset adapter is new (our packed HDF5s instead of the mirdata `mazurka_h5` index)
- Env: `beat_mir` (TF 2.15 + Keras 2.15 + tensorflow_addons + madmom + mir_eval). Recreate via `pip install -r beat_tcn/requirements.txt` if missing.

### Vienna4x22
| Fold | Best-val ckpt: Beat F1 | Best-val ckpt: Downbeat F1 | Final ckpt: Beat F1 | Final ckpt: Downbeat F1 |
|-----:|-----------------------:|---------------------------:|--------------------:|------------------------:|
| 0 | 0.5332 | 0.2210 | 0.5183 | 0.2209 |
| 1 | **0.5838** | 0.2288 | **0.5726** | 0.2295 |
| 2 | 0.5726 | 0.2276 | 0.5697 | 0.2270 |
| 3 | 0.5431 | 0.2290 | 0.5485 | **0.2350** |
| 4 | 0.5221 | **0.2297** | 0.5133 | 0.2297 |
| **Avg ± Std** | **0.5510 ± 0.0235** | **0.2272 ± 0.0032** | **0.5445 ± 0.0249** | **0.2284 ± 0.0046** |
| **Best beat fold = 1** | **0.5838** | **0.2288** | **0.5726** | **0.2295** |

### Batik-plays-Mozart
| Fold | Best-val ckpt: Beat F1 | Best-val ckpt: Downbeat F1 | Final ckpt: Beat F1 | Final ckpt: Downbeat F1 |
|-----:|-----------------------:|---------------------------:|--------------------:|------------------------:|
| 0 | 0.2292 | 0.1203 | **0.2310** | **0.1239** |
| 1 | 0.2247 | **0.1212** | 0.2275 | 0.1202 |
| 2 | 0.2284 | 0.1197 | 0.2289 | 0.1193 |
| 3 | 0.2292 | 0.1185 | 0.2297 | 0.1185 |
| 4 | **0.2310** | 0.1206 | 0.2290 | 0.1219 |
| **Avg ± Std** | **0.2285 ± 0.0021** | **0.1201 ± 0.0009** | **0.2292 ± 0.0011** | **0.1208 ± 0.0019** |
| **Best beat fold = 4** | **0.2310** | **0.1206** | **0.2290** | **0.1219** |

## 0. Setup

In [1]:
import os, warnings
warnings.filterwarnings('ignore')
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'
import numpy as np, h5py
from pathlib import Path
from tqdm import tqdm
import tensorflow as tf, keras
from keras.utils import Sequence
import keras.backend as K
from keras.models import Model
from keras.layers import (Input, Dense, Activation, Conv1D, Conv2D, MaxPooling2D,
                          Reshape, Dropout, SpatialDropout1D, Concatenate, Add)
import madmom
from madmom.processors import SequentialProcessor
from madmom.audio.signal import SignalProcessor, FramedSignalProcessor
from madmom.audio.stft import ShortTimeFourierTransformProcessor
from madmom.audio.spectrogram import FilteredSpectrogramProcessor, LogarithmicSpectrogramProcessor

for gpu in tf.config.list_physical_devices('GPU'):
    try: tf.config.experimental.set_memory_growth(gpu, True)
    except Exception: pass

# Mazurka-trained ckpts (5 folds)
TCN_CKPT_BASE = "/media/datadisk/home/22828187/zhanh/202509_icassp_data/workspaces/mazurka_ckpts/ckpt_baselines/beat_tcn_ckpt/checkpoint/fold{f}_eps120"

# Packed test HDF5s (from CrossEval.ipynb / pack_h5_{vienna,batik})
TEST_WS = "/media/datadisk/home/22828187/zhanh/202509_bark_data/workspaces"
VIENNA_H5 = f"{TEST_WS}/hdf5s/vienna_sr22050"
BATIK_H5  = f"{TEST_WS}/hdf5s/batik_sr22050"

2026-05-18 02:25:15.347585: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2026-05-18 02:25:15.347616: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-05-18 02:25:15.348592: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


## 1. Adapter — load our packed HDF5s as mirdata-style tracks
Exposes `t.audio = (waveform_float32, sr)`, `t.beats.times`, `t.beats.positions` so the original `DataSequence` and `predict_beats_downbeats` work unchanged.

In [2]:
class _H5Track:
    def __init__(self, h5_path, sr=22050):
        self._sr = sr
        with h5py.File(h5_path, 'r') as f:
            self._wav = (f['waveform'][:].astype(np.float32) / 32768.0)
            beats     = f['beat_time'][:].astype(np.float64)
            downbeats = f['downbeat_time'][:].astype(np.float64)
        positions = np.full(len(beats), 2, dtype=int)
        for d in downbeats:
            positions[int(np.argmin(np.abs(beats - d)))] = 1
        class _B: pass
        self.beats = _B(); self.beats.times = beats; self.beats.positions = positions
    @property
    def audio(self):
        return self._wav, self._sr

def load_tracks_from_h5_dir(h5_root):
    return {f"{p.parent.name}/{p.stem}": _H5Track(str(p))
            for p in sorted(Path(h5_root).rglob('*.h5'))}

## 2. TCN model + preprocessor + DBN inference
Copied verbatim from `Eval_BeatTCN_MazurkaBL.ipynb` (sections 4 — model definition, audio preprocessor, predict / evaluate).

In [3]:
# --- TCN model ---
def _residual_block(x, i, activation, num_filters, kernel_size, padding, dropout_rate=0, name=''):
    name = f"{name}_dilation_{i}"
    res_x = Conv1D(num_filters, 1, padding='same', name=name + '_1x1_conv_residual')(x)
    conv_1 = Conv1D(num_filters, kernel_size, dilation_rate=i,   padding=padding, name=name + '_dilated_conv_1')(x)
    conv_2 = Conv1D(num_filters, kernel_size, dilation_rate=2*i, padding=padding, name=name + '_dilated_conv_2')(x)
    concat = Concatenate(name=name + '_concat')([conv_1, conv_2])
    x = Activation(activation, name=name + '_activation')(concat)
    x = SpatialDropout1D(dropout_rate, name=f"{name}_spatial_dropout_{dropout_rate}")(x)
    x = Conv1D(num_filters, 1, padding='same', name=name + '_1x1_conv')(x)
    return Add(name=name + '_merge_residual')([res_x, x]), x

class _TCN:
    def __init__(self, num_filters, kernel_size, dilations, activation='elu', padding='same', dropout_rate=0.15, name='tcn'):
        self.num_filters, self.kernel_size, self.dilations = num_filters, kernel_size, dilations
        self.activation, self.padding, self.dropout_rate, self.name = activation, padding, dropout_rate, name
    def __call__(self, inputs):
        x, skips = inputs, []
        for i, nf in zip(self.dilations, self.num_filters):
            x, s = _residual_block(x, i, self.activation, nf, self.kernel_size, self.padding, self.dropout_rate, name=self.name)
            skips.append(s)
        x = Activation(self.activation, name=self.name + '_activation')(x)
        return x, Add(name=self.name + '_merge_skip_connections')(skips)

def create_model(input_shape, num_filters=20, num_dilations=11, kernel_size=5, activation='elu', dropout_rate=0.15):
    inp = Input(shape=input_shape)
    c = Conv2D(num_filters, (3, 3), padding='valid', name='conv_1_conv')(inp)
    c = Activation(activation, name='conv_1_activation')(c)
    c = MaxPooling2D((1, 3), name='conv_1_max_pooling')(c)
    c = Dropout(dropout_rate, name='conv_1_dropout')(c)
    c = Conv2D(num_filters, (1, 10), padding='valid', name='conv_2_conv')(c)
    c = Activation(activation, name='conv_2_activation')(c)
    c = MaxPooling2D((1, 3), name='conv_2_max_pooling')(c)
    c = Dropout(dropout_rate, name='conv_2_dropout')(c)
    c = Conv2D(num_filters, (3, 3), padding='valid', name='conv_3_conv')(c)
    c = Activation(activation, name='conv_3_activation')(c)
    c = MaxPooling2D((1, 3), name='conv_3_max_pooling')(c)
    c = Dropout(dropout_rate, name='conv_3_dropout')(c)
    x = Reshape((-1, num_filters), name='tcn_input_reshape')(c)
    dilations = [2 ** i for i in range(num_dilations)]
    tcn, _ = _TCN([num_filters]*len(dilations), kernel_size, dilations, activation, 'same', dropout_rate)(x)
    b = Dropout(dropout_rate, name='beats_dropout')(tcn);     b = Dense(1, name='beats_dense')(b);     b = Activation('sigmoid', name='beats')(b)
    d = Dropout(dropout_rate, name='downbeats_dropout')(tcn); d = Dense(1, name='downbeats_dense')(d); d = Activation('sigmoid', name='downbeats')(d)
    return Model(inp, outputs=[b, d])

# --- Audio preprocessor (matches what mazurka_h5 was packed with) ---
FPS, FFT_SIZE, SAMPLE_RATE, NUM_BANDS, MASK_VALUE = 50, 1024, 22050, 12, -1
class PreProcessor(SequentialProcessor):
    def __init__(self, frame_size=FFT_SIZE, num_bands=NUM_BANDS, log=np.log, add=1e-6, fps=FPS):
        super().__init__((SignalProcessor(num_channels=1, sample_rate=SAMPLE_RATE),
                          FramedSignalProcessor(frame_size=frame_size, fps=fps),
                          ShortTimeFourierTransformProcessor(),
                          FilteredSpectrogramProcessor(num_bands=num_bands),
                          LogarithmicSpectrogramProcessor(log=log, add=add),
                          np.array))
        self.fps = fps

def _cnn_pad(data, pad_frames):
    return np.concatenate((np.repeat(data[:1], pad_frames, axis=0), data,
                           np.repeat(data[-1:], pad_frames, axis=0)))

# --- 30 s segment sequence (beat/downbeat only, no tempo head) ---
class DataSequence(Sequence):
    def __init__(self, tracks, pre_processor, pad_frames=2, win_s=30.0, hop_s=30.0):
        self.fps = pre_processor.fps
        self.win = int(round(win_s * self.fps)); self.hop = int(round(hop_s * self.fps))
        self.pad = pad_frames
        self.X, self.Yb, self.Yd, self.segs = {}, {}, {}, []
        for key, t in tracks.items():
            y, sr = t.audio
            X = pre_processor(madmom.audio.Signal(y, sr)).astype('float32'); T = len(X)
            bs = t.beats.times
            beat     = madmom.utils.quantize_events(bs, fps=self.fps, length=T).astype('float32')
            downbeat = madmom.utils.quantize_events(bs[t.beats.positions.astype(int) == 1],
                                                    fps=self.fps, length=T).astype('float32')
            self.X[key], self.Yb[key], self.Yd[key] = X, beat, downbeat
            # Cover short clips with a single window starting at 0
            if T < self.win:
                self.segs.append((key, 0, T))
            else:
                for s in range(0, T - self.win + 1, self.hop):
                    self.segs.append((key, s, s + self.win))
        self.N = len(self.segs)
    def __len__(self): return self.N
    def __getitem__(self, i):
        key, a, b = self.segs[i]
        x = self.X[key][a:b]
        if self.pad: x = _cnn_pad(x, self.pad)
        y = {'beats': self.Yb[key][a:b][None, ..., None], 'downbeats': self.Yd[key][a:b][None, ..., None]}
        return x[None, ..., None], y

# --- DBN inference + evaluation (same DBN params as Mazurka eval) ---
def predict_beats_downbeats(model, dataset, fps=FPS, dedup_frames=2, desc="Predicting"):
    dedup_sec = dedup_frames / float(fps)
    def _bt(act):
        return madmom.features.beats.DBNBeatTrackingProcessor(
            min_bpm=90.0, max_bpm=215.0, fps=fps, transition_lambda=100, threshold=0.05)(act)
    def _dt(b_act, d_act):
        c = np.vstack([np.maximum(b_act - d_act, 0.0), d_act]).T
        out = madmom.features.downbeats.DBNDownBeatTrackingProcessor(
            beats_per_bar=[3], min_bpm=90.0, max_bpm=215.0, fps=fps, transition_lambda=100)(c)
        return out[:, 0] if len(out) else np.empty((0,), dtype=float)
    def _merge(lst):
        if not lst: return np.empty((0,), dtype=float)
        t = np.sort(np.concatenate(lst))
        keep = [t[0]]
        for x in t[1:]:
            if x - keep[-1] >= dedup_sec: keep.append(x)
        return np.asarray(keep)
    bbuf, dbuf = {}, {}
    for i in tqdm(range(len(dataset)), desc=desc):
        k, a, _ = dataset.segs[i]
        off = a / float(fps)
        x, _ = dataset[i]
        b_act, d_act = model.predict(x, verbose=0)
        b_act, d_act = b_act.squeeze(), d_act.squeeze()
        bbuf.setdefault(k, []).append(_bt(b_act) + off)
        dbuf.setdefault(k, []).append(_dt(b_act, d_act) + off)
    return {k: {'beats': _merge(bbuf[k]), 'downbeats': _merge(dbuf.get(k, []))} for k in bbuf}

def evaluate_beats_and_downbeats(detections, beat_ann, downbeat_ann):
    bev = [madmom.evaluation.beats.BeatEvaluation(d['beats'], beat_ann[k])
           for k, d in detections.items() if k in beat_ann]
    dev = [madmom.evaluation.beats.BeatEvaluation(d['downbeats'], downbeat_ann[k], downbeats=True)
           for k, d in detections.items() if k in downbeat_ann]
    bm = madmom.evaluation.beats.BeatMeanEvaluation(bev) if bev else None
    dm = madmom.evaluation.beats.BeatMeanEvaluation(dev) if dev else None
    return {'beat': bm, 'downbeat': dm}

## 3. Cross-corpus runner — 5 mazurka folds × {best, final} on one corpus

In [4]:
def run_corpus(h5_root, label, pad_frames=2):
    print(f"\n===== {label}  ({h5_root}) =====")
    tracks = load_tracks_from_h5_dir(h5_root)
    print(f"loaded {len(tracks)} tracks")
    beat_ann     = {k: t.beats.times for k, t in tracks.items()}
    downbeat_ann = {k: t.beats.times[t.beats.positions.astype(int) == 1] for k, t in tracks.items()}

    pp = PreProcessor()
    test = DataSequence(tracks, pre_processor=pp, pad_frames=pad_frames)
    input_shape = (None,) + test[0][0].shape[-2:]
    model = create_model(input_shape)

    summary = []
    for f in range(5):
        ckdir = Path(TCN_CKPT_BASE.format(f=f))
        for tag, fname in [('BEST', 'model_best.h5'), ('FINAL', 'model_final.h5')]:
            ckpt = ckdir / fname
            if not ckpt.is_file():
                print(f"[skip] missing {ckpt}"); continue
            model.load_weights(str(ckpt))
            det = predict_beats_downbeats(model, test, fps=pp.fps, desc=f"fold{f} {tag}")
            sc = evaluate_beats_and_downbeats(det, beat_ann, downbeat_ann)
            b = float(sc['beat'].fmeasure) if sc['beat'] else float('nan')
            d = float(sc['downbeat'].fmeasure) if sc['downbeat'] else float('nan')
            print(f"fold {f} [{tag:5s}] Beat F1: {b:.4f} | Downbeat F1: {d:.4f}")
            summary.append((f, tag, b, d))
    return summary

---
## 4. Vienna4x22

In [5]:
vienna_results = run_corpus(VIENNA_H5, label="Vienna4x22")


===== Vienna4x22  (/media/datadisk/home/22828187/zhanh/202509_bark_data/workspaces/hdf5s/vienna_sr22050) =====
loaded 88 tracks


fold0 BEST: 100%|██████████| 227/227 [00:12<00:00, 17.76it/s]


fold 0 [BEST ] Beat F1: 0.5332 | Downbeat F1: 0.2210


fold0 FINAL: 100%|██████████| 227/227 [00:11<00:00, 19.70it/s]


fold 0 [FINAL] Beat F1: 0.5183 | Downbeat F1: 0.2209


...

fold 3 [FINAL] Beat F1: 0.5485 | Downbeat F1: 0.2350


fold4 BEST: 100%|██████████| 227/227 [00:11<00:00, 19.45it/s]


fold 4 [BEST ] Beat F1: 0.5221 | Downbeat F1: 0.2297


fold4 FINAL: 100%|██████████| 227/227 [00:11<00:00, 19.33it/s]


fold 4 [FINAL] Beat F1: 0.5133 | Downbeat F1: 0.2297


---
## 5. Batik-plays-Mozart

In [6]:
batik_results = run_corpus(BATIK_H5, label="Batik-plays-Mozart")


===== Batik-plays-Mozart  (/media/datadisk/home/22828187/zhanh/202509_bark_data/workspaces/hdf5s/batik_sr22050) =====
loaded 36 tracks


fold0 BEST: 100%|██████████| 436/436 [00:22<00:00, 19.12it/s]


fold 0 [BEST ] Beat F1: 0.2292 | Downbeat F1: 0.1203


fold0 FINAL: 100%|██████████| 436/436 [00:22<00:00, 19.25it/s]


fold 0 [FINAL] Beat F1: 0.2310 | Downbeat F1: 0.1239


...

fold 3 [FINAL] Beat F1: 0.2297 | Downbeat F1: 0.1185


fold4 BEST: 100%|██████████| 436/436 [00:22<00:00, 19.54it/s]


fold 4 [BEST ] Beat F1: 0.2310 | Downbeat F1: 0.1206


fold4 FINAL: 100%|██████████| 436/436 [00:23<00:00, 18.91it/s]


fold 4 [FINAL] Beat F1: 0.2290 | Downbeat F1: 0.1219
